# PlantVillage Disease Classification

<p align="right">
Run Time: ~20 minutes with training included / ~15 minutes with training skipped
</p>

This notebook walks through the complete pipeline to train, quantize, convert, and
benchmark an AkidaNet model on the **PlantVillage** dataset for Akida 1 hardware.

PlantVillage contains 54,306 images of healthy and diseased plant leaves across
38 categories (14 crop species × multiple disease types plus healthy variants).
The task is a 38-class image classification problem: given a 224×224 RGB image of
a leaf, identify the crop species and disease (or healthy state). The pipeline loads
54,305 of these images (a few files are not recognised as valid images and are skipped).

The pipeline follows the standard Akida workflow:
1. Train a float model
2. Post-training quantization (PTQ)
3. Quantization-aware training (QAT) fine-tuning
4. Conversion to Akida `.fbz` format
5. Hardware evaluation and benchmarking

In [1]:
# Colab-only setup. Local users: ignore this cell — it does nothing for you.
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('colab_setup.py'):
        !wget -q https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/akida1/model_zoo/plant_village/colab_setup.py
    import colab_setup; colab_setup.setup()


In [4]:
import os
import numpy as np
import tensorflow as tf

import pooch
from tf_keras.utils import set_random_seed

from cnn2snn import load_quantized_model

DATA_PATH = './data/plant_village'
MODELS_DIR = './models/'
os.makedirs(MODELS_DIR, exist_ok=True)

RUN_FLOAT_TRAINING = True
RUN_QAT_TRAINING = True

SEED = 42

# Must be called before any TF ops to make GPU ops (conv backward passes,
# bilinear resize, etc.) deterministic. Has a small throughput cost.
tf.config.experimental.enable_op_determinism()

## Dataset

The **PlantVillage** dataset is loaded from the authors' official GitHub repository
([spMohanty/PlantVillage-Dataset](https://github.com/spMohanty/PlantVillage-Dataset)).
On the first run, the images are downloaded and extracted to `DATA_PATH`
automatically; subsequent runs read from the local copy.

The dataset is split 80/10/10 (train/val/test). Images are resized from variable
original sizes to **224 × 224 RGB** and delivered as uint8 pixel values (0–255).
Training applies random horizontal flip, brightness jitter, and contrast jitter
for regularisation.

The download and extraction are handled inside `get_data()` on first call, so
there is no separate pre-download step required.


In [ ]:
from plant_village_data import get_data

INPUT_SHAPE = (224, 224, 3)
BATCH_SIZE = 32

train_ds, val_ds, test_ds = get_data(DATA_PATH, input_shape=INPUT_SHAPE, batch_size=BATCH_SIZE, seed=SEED)
print('Datasets ready.')

KeyboardInterrupt: 

## Model

The model is based on **AkidaNet** (`akida_models.akidanet_imagenet`) with:
- Width multiplier **alpha = 0.5** — provides sufficient capacity for 38 classes
  while remaining efficient on Akida 1 hardware
- Input resolution **224 × 224 RGB**
- **38-class** classification head (replacing the ImageNet top)
- **Input scaling (255, 0)** built into the model — the pipeline delivers raw
  uint8 pixel values and the model normalises them internally

AkidaNet is specifically designed for Akida hardware: it uses only operations
that map efficiently to Akida Neural Processors (NPs), including depthwise
separable convolutions and ReLU activations.

In [7]:
from plant_village_model import build_plant_village_model

model = build_plant_village_model(seed=SEED)
model.summary()

/usr/local/lib/python3.12/dist-packages/akida_models/model_io.py:147: UserWarning: Model akidanet_imagenet_224_alpha_50.h5 has been trained with akida_models 1.1.10 which is the last version supporting 1.0 models training. Continuing execution.
  warnings.warn(f'Model {model_name_v1} has been trained with akida_models 1.1.10 which is '


5697512/5697512 [==============================] - 1s 0us/step
Download complete.


Model: "akidanet_plantvillage"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input (InputLayer)          [(None, 224, 224, 3)]     0         
                                                                 
 rescaling (Rescaling)       (None, 224, 224, 3)       0         
                                                                 
 conv_0 (Conv2D)             (None, 112, 112, 16)      432       
                                                                 
 conv_0/BN (BatchNormalizat  (None, 112, 112, 16)      64        
 ion)                                                            
                                                                 
 conv_0/relu (ReLU)          (None, 112, 112, 16)      0         
                                                                 
 conv_1 (Conv2D)             (None, 112, 112, 32)      4608      
                                             

## Float Training

The model is trained in full float32 precision for 10 epochs using the Adam
optimiser and sparse categorical cross-entropy loss (with `from_logits=True`,
since the model head outputs raw logits rather than softmax probabilities).

The learning rate follows an **exponential decay** schedule, starting at `1e-3`
and decaying to approximately `1e-5` by the final epoch.

Set `RUN_FLOAT_TRAINING = True` above to train from scratch. Otherwise, the
cell below loads a pre-trained float model from the `pretrained_models/` folder.

In [1]:
from plant_village_train import train_plant_village

if RUN_FLOAT_TRAINING:
    train_plant_village(
        model, train_ds, val_ds,
        epochs=10,
        learning_rate=1e-3,
        seed=SEED)
    model.save(
        MODELS_DIR + 'akidanet_plant_village.h5',
        include_optimizer=False)
    print('Float model saved.')
else:
    float_model_path = 'pretrained_models/akidanet_plant_village.h5'
    model = load_quantized_model(float_model_path)
    model.compile(metrics=['accuracy'])

ModuleNotFoundError: No module named 'plant_village_train'

In [ ]:
_, float_acc = model.evaluate(test_ds, verbose=1)
print(f'Float accuracy: {float_acc:.4f}')

170/170 [==============================] - 46s 47ms/step - loss: 0.1423 - accuracy: 0.9982
Float accuracy: 0.9982


## Quantization

Post-training quantization (PTQ) via `cnn2snn.quantize` converts the model
to fixed-point arithmetic:
- **Input**: 8-bit (`-i 8`)
- **Weights**: 4-bit (`-w 4`)
- **Activations**: 4-bit (`-a 4`)

4-bit quantization must be used to be compatible with Akida 1 hardware. Note though that
the first layer (both its inputs and weights) can be 8-bit.

In [ ]:
import cnn2snn

quantized_model = cnn2snn.quantize(
    model,
    input_weight_quantization=8,
    weight_quantization=4,
    activ_quantization=4)
print('Model quantized to i8/w4/a4.')

Model quantized to i8/w4/a4.


Quantizing a model after training like this is referred to as Post-Training
Quantization (PTQ). It can slightly reduce accuracy (especially at 4-bits as
here) because the model was trained with continuous weights but is now
evaluated with discrete values.

In [ ]:
quantized_model.compile(metrics=['accuracy'])
_, ptq_acc = quantized_model.evaluate(test_ds, verbose=1)
print(f'PTQ accuracy: {ptq_acc:.4f}')

170/170 [==============================] - 48s 36ms/step - loss: 0.3775 - accuracy: 0.9958
PTQ accuracy: 0.9958


## Quantization-Aware Training (QAT)

We can run Quantization Aware Training (QAT) to recover most of the drop in
accuracy. QAT fine-tunes the quantized model for a few epochs (here, 2) at a
reduced learning rate (`1e-4`). Note that, although it can sound intimidating,
QAT with BrainChip's quantization tools is no more complex than simply sending
the quantized model back through the same training pipeline that was used to
prepare the float model in the first place.

Set `RUN_QAT_TRAINING = True` above to run QAT locally. Otherwise, the cell
below loads a pre-trained QAT model from the `pretrained_models/` folder.

In [ ]:
if RUN_QAT_TRAINING:
    # We refetch the dataset, only to ensure reproducibility against the non-notebook pipeline.
    # This resets the shuffle seed on the training data
    train_ds, val_ds, test_ds = get_data(DATA_PATH, input_shape=INPUT_SHAPE, batch_size=BATCH_SIZE, seed=SEED)
    train_plant_village(
        quantized_model, train_ds, val_ds,
        epochs=2,
        learning_rate=1e-4)
    quantized_model.save(
        MODELS_DIR + 'akidanet_plant_village_qat.h5',
        include_optimizer=False)
    print('QAT model saved.')
else:
    qat_model_path = 'pretrained_models/akidanet_plant_village_qat.h5'
    quantized_model = load_quantized_model(qat_model_path)
    quantized_model.compile(metrics=['accuracy'])

Found 54305 files belonging to 38 classes.
Epoch 1/2
1358/1358 [==============================] - 174s 123ms/step - loss: 0.4390 - accuracy: 0.9836 - val_loss: 0.4450 - val_accuracy: 0.9814 - lr: 1.0000e-04
Epoch 2/2
1358/1358 [==============================] - 167s 121ms/step - loss: 0.3851 - accuracy: 0.9982 - val_loss: 0.3910 - val_accuracy: 0.9967 - lr: 1.0000e-05
QAT model saved.


In [ ]:
_, qat_acc = quantized_model.evaluate(test_ds, verbose=1)
print(f'QAT accuracy: {qat_acc:.4f}')

170/170 [==============================] - 45s 43ms/step - loss: 0.3912 - accuracy: 0.9956
QAT accuracy: 0.9956


## Conversion to Akida Format

`cnn2snn.convert` compiles the quantized Keras model into an Akida `.fbz`
model that can be loaded and executed directly on AKD1500 hardware.
The converter verifies hardware compatibility and maps each layer to its
corresponding Akida primitive.

In [ ]:
akida_model = cnn2snn.convert(quantized_model)

akida_model_path = os.path.join(MODELS_DIR, 'akidanet_plant_village_qat.fbz')
akida_model.save(akida_model_path)
print(f'Akida model saved to {akida_model_path}')
akida_model.summary()

Akida model saved to ./models/akidanet_plant_village_qat.fbz
                 Model Summary                  
________________________________________________
Input shape    Output shape  Sequences  Layers
[224, 224, 3]  [1, 1, 38]    1          16    
________________________________________________

____________________________________________________________
Layer (type)              Output shape    Kernel shape    

============= SW/conv_0-predictions (Software) =============

conv_0 (InputConv.)       [112, 112, 16]  (3, 3, 3, 16)   
____________________________________________________________
conv_1 (Conv.)            [112, 112, 32]  (3, 3, 16, 32)  
____________________________________________________________
conv_2 (Conv.)            [56, 56, 64]    (3, 3, 32, 64)  
____________________________________________________________
conv_3 (Conv.)            [56, 56, 64]    (3, 3, 64, 64)  
____________________________________________________________
separable_4 (Sep.Conv.)   [28, 28,

## Evaluation of Akida Model

We now run evaluation through the Akida model, to check that accuracy is
comparable to that obtained from the quantized tf_keras model. Here, we deliberately
use the software backend (the default, since we do not check for and map to
a connected hardware device): this delivers a
bit-accurate simulation of the results that will be obtained when running
the model on hardware.

In the accompanying [plant_village_notebook_benchmark.ipynb](plant_village_notebook_benchmark.ipynb)
the same evaluation is run using the hardware backend (if, of course, a hardware Akida
device is connected), allowing you to confirm that the results are identical.

### Run Evaluation on Akida

The Akida runtime cannot consume `tf.data.Dataset` objects directly, rather
it expects a 4D numpy array (n, h, w, c) in uint8 format. So we
iterate over test batches manually.

The model output tensor has shape `(B, 1, 1, C)` which is squeezed to
`(B, C)` before taking the class argmax.

In [ ]:
from tqdm import tqdm

labels_all = []
logits_all = []
for batch, label_batch in tqdm(test_ds, desc="Evaluating on Akida"):
    if not isinstance(batch, np.ndarray):
        batch = batch.numpy()

    logits_batch = akida_model.predict(batch, batch_size=BATCH_SIZE)

    logits_batch = logits_batch.squeeze(axis=(1, 2))
    labels_all.append(label_batch)
    logits_all.append(logits_batch)

labels_all = np.concatenate(labels_all)
logits_all = np.concatenate(logits_all)
preds = np.argmax(logits_all, axis=1)

akida_acc = float(np.mean(preds == np.array(labels_all)))
print(f'Akida accuracy: {akida_acc:.4f}')

Evaluating on Akida: 100%|██████████| 170/170 [11:21<00:00,  4.01s/it]

Akida accuracy: 0.9958


### Activation Sparsity

Akida hardware skips computation for zero-valued activations, so activation
sparsity directly reduces both energy consumption and inference latency.
Below we measure per-layer sparsity on a 100-sample calibration batch drawn
from the training set.

In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity
from plant_village_data import get_samples

NUM_SAMPLES = 100

samples = get_samples(DATA_PATH, input_shape=INPUT_SHAPE, num_samples=NUM_SAMPLES)
sparsity_dict = compute_sparsity(akida_model, samples=samples)
pretty_print_sparsity(sparsity_dict)

Found 54305 files belonging to 38 classes.

Layer            Sparsity
-------------------------
conv_0            31.36%
conv_1            48.62%
conv_2            56.30%
conv_3            42.13%
separable_4       41.81%
separable_5       64.38%
separable_6       44.93%
separable_7       56.36%
separable_8       62.96%
separable_9       67.92%
separable_10      76.30%
separable_11      61.41%
separable_12      88.47%
separable_13      62.89%
fc_1              62.24%
predictions        0.00%
-------------------------
Mean              54.26%


## Summary

The table below compares test accuracy across the three model variants.
The goal is that QAT and Akida accuracy remain close to the float baseline.

In [ ]:
print('PlantVillage results')
print('=' * 40)
print(f'  Float accuracy:     {float_acc * 100:.2f}%')
print(f'  QAT accuracy:       {qat_acc * 100:.2f}%')
print(f'  Akida accuracy:     {akida_acc * 100:.2f}%')

PlantVillage results
  Float accuracy:     99.82%
  QAT accuracy:       99.56%
  Akida accuracy:     99.58%
